# BTI 원료별 매출 추적 분석

> 원료(소재) 기준으로 제품 매출을 추적하여 팀별 매출 기여도를 분석합니다.
>
> **파라미터 변경 시**: 아래 "분석 파라미터" 셀의 값만 수정하면 됩니다.

In [ ]:
!pip install awswrangler openpyxl xlsxwriter

In [ ]:
import awswrangler as wr
import pandas as pd
import numpy as np
import re
import logging
from collections import defaultdict, deque

logger = logging.getLogger()

def query2athena(query, database='data_mart'):
    """Athena 쿼리 실행"""
    try:
        df = wr.athena.read_sql_query(query, database=database)
        return df
    except Exception as e:
        msg = query.replace('\n', ' ')
        logger.info(f'Failed to query: {msg}')
        raise e

In [ ]:
# ============================================================
#  분석 파라미터 (팀/기간/파일 변경 시 이 블록만 수정)
# ============================================================
COMPID     = "1200"                                  # 법인 코드
T_DEPT     = "%MB2%"                                 # 타겟 부서 패턴 (LIKE 조건)
START_DATE = "2025-01-01"                            # 매출 조회 시작일
END_DATE   = "2025-03-31"                            # 매출 조회 종료일

INPUT_RAW_FILE = "20260205_MB2팀 소재 리스트.xlsx"     # 팀 관리소재 리스트
OUTPUT_FILE    = "251Q_MB2_revenue_report.xlsx"       # 최종 결과 파일명

# ============================================================
#  WHERE 조건 생성 유틸
# ============================================================
def build_where(
    compid=None, t_dept=None, start_date=None, end_date=None, *,
    compid_col="compid", dept_col="dept_nm", date_col="first_in", table_alias=None,
):
    conds = []
    prefix = f"{table_alias}." if table_alias else ""
    if compid:
        conds.append(f"{prefix}{compid_col} = '{compid}'")
    if t_dept:
        conds.append(f"{prefix}{dept_col} LIKE '{t_dept}'")
    if start_date and end_date:
        conds.append(
            f"date({prefix}{date_col}) BETWEEN DATE '{start_date}' AND DATE '{end_date}'"
        )
    return "" if not conds else "1=1\n  AND " + "\n  AND ".join(conds)

## 1. 타겟 유저 & 소재 조회

In [6]:
# where 조건 생성
where_sql = build_where(compid=COMPID,t_dept=T_DEPT, compid_col="comp_id")

q=f'''
select 
       user_id
      ,user_nm
      ,ad_upn as user_email
      ,dept_nm
from data_mart.t_dmart_employee_info
where {where_sql}
'''

#타겟 유저 정보 (MB랩 전체) -> 260129시점 14명
t_user_info_df = query2athena(q)

In [7]:
t_user_info_df

,user_id,user_nm,user_email,dept_nm
0,112210016,허영목,ymheo@cosmax.com,[BTI-연구소-비제형]MB2팀
1,112220050,이하은,haeun.lee@cosmax.com,[BTI-연구소-비제형]MB2팀
2,112230063,천세진,sejin.cheon@cosmax.com,[BTI-연구소-비제형]MB2팀
3,112240072,김기욱,giuk.kim@cosmax.com,[BTI-연구소-비제형]MB2팀
4,112240077,우아람,aram.woo@cosmax.com,[BTI-연구소-비제형]MB2팀


In [ ]:
# 팀 관리소재 리스트 로드
t_user_raw_df = pd.read_excel(INPUT_RAW_FILE)
t_user_raw_df = (t_user_raw_df[['품목코드', '품목명', '상태', '연구개발담당']]
                 .rename(columns={'품목코드': 'raw_cd', '품목명': 'raw_nm', '상태': 'mmsta', '연구개발담당': 'raw_user_id'}))
t_user_raw_df['raw_cd'] = t_user_raw_df['raw_cd'].astype(str)

In [10]:
t_user_raw_df

,raw_cd,raw_nm,mmsta,raw_user_id
0,6041324,(CITES)L'ENIGME(Z2),Z2,112210016
1,6046718,(CITES)OPUNTIA BIOCOMPLEX SH,NO,112210016
2,6041312,ABSOLUTE S(중국사용금지)(Z2),Z2,112210016
3,6046429,DERMA-MU,NO,112210016
4,6046342,DR. TASER(H),NO,112210016
...,...,...,...,...
113,6053436,MUTE-OLL,NO,112240077
114,6053329,PORPHYRIDAN(C),NO,112240077
115,6050939,PORPHYRIDAN(H),NO,112240077
116,6050421,(누797)UNTOPINOL(C)_PPD100,NO,112240072


## 2. BOM 조회 & Closure 생성

In [12]:
# where 조건 생성
where_sql = build_where(compid=COMPID)
 
q=f'''
SELECT DISTINCT
  CAST(spcomp AS varchar) AS child_code,
  CAST(mitem  AS varchar) AS parent_code,
  spqty
FROM cai.t_rd_bom
WHERE {where_sql}
  AND spcomp IS NOT NULL
  AND mitem  IS NOT NULL
'''
 
#bom(함량포함) df 가져오기
t_bom_edges_df = query2athena(q)

In [13]:
#함량정보 0인 정보 제거
t_bom_edges_df = t_bom_edges_df[t_bom_edges_df['spqty'] != 0].copy()

In [14]:
t_bom_edges_df

,child_code,parent_code,spqty
0,7BAC000121701,2DZK000071010,1.000000000000
1,6029817,TA0000867,0.350000000000
2,6010324,3ELC00804190,5.000000000000
3,6008572,ID0000985,0.300000000000
4,6010464,3CR095095617,0.800000000000
...,...,...,...
5051509,6012325,3CST00537110,0.001000000000
5051510,6000819,3SHD00097130,7.330000000000
5051511,6036452,3KRT00505110,0.120000000000
5051512,6040001,3NDA01268140,7.000000000000


In [ ]:
# parent_map: child → [(parent, qty), ...] 매핑
parent_map = defaultdict(list)

for c, p, q in zip(
    t_bom_edges_df["child_code"].astype(str),
    t_bom_edges_df["parent_code"].astype(str),
    t_bom_edges_df["spqty"]
):
    parent_map[c].append((p, q))

In [ ]:
def build_closure_for_targets(target_codes, parent_map, max_depth=50):
    """타겟 원료 코드들의 BOM closure 생성 (BFS)"""
    rows = []

    for child in map(str, target_codes):
        q = deque([(child, 0, None)])
        visited = set([child])

        while q:
            node, d, spqty = q.popleft()

            for parent, parent_spqty in parent_map.get(node, []):
                if parent in visited:
                    continue

                visited.add(parent)
                depth = d + 1
                child_spqty = parent_spqty if d == 0 else spqty

                rows.append((child, parent, depth, child_spqty))

                if depth < max_depth:
                    q.append((parent, depth, child_spqty))

    return pd.DataFrame(
        rows,
        columns=["child_code", "ancestor_code", "depth", "child_spqty"]
    )

# 타겟 코드 closure 생성
target_codes = t_user_raw_df['raw_cd'].astype(str).tolist()
closure_df = build_closure_for_targets(target_codes, parent_map)

In [17]:
closure_df

,child_code,ancestor_code,depth,child_spqty
0,6046718,3CR098800063,1,0.100000000000
1,6046718,3CI000002473,1,0.500000000000
2,6046718,3CR098500291,1,1.00000E-7
3,6046718,3DNG00001110,1,1.00000E-7
4,6046718,9DNG0000110,2,1.00000E-7
...,...,...,...,...
1833,6051094,RAC0000665,1,0.312500000000
1834,6051094,3CR099700001,1,0.400000000000
1835,6051094,3CR099700002,1,0.800000000000
1836,6050939,3CR099700001,1,0.100000000000


## 3. 매출 데이터 조회 & 전처리

In [48]:
# where 조건 생성
where_sql = build_where(compid=COMPID,start_date=START_DATE,end_date=END_DATE, compid_col="comp_id", date_col="base_time", table_alias="a")

q=f'''
select 
     a.mitem_code
    ,a.mitem_name
    ,b.customer_code
    ,b.customer_name
    ,a.sales_quantity
    ,a.total_revenue
    ,a.product_sales_revenue
    ,a.net_revenue
    ,a.currency
    ,a.category
    ,a.unit
    ,a.forml_code
    ,a.forml_name
    ,a.base_time
from data_mart.t_dmart_gcc_monthly_sales_revenue a
left join data_mart.t_dmart_master_customer b
       on a.customer_code = b.customer_code
where {where_sql}
  and b.comp_id = '1200'   -- 법인별 중복 고객사 있음
  and a.mitem_code like '9%'  --9자코드 완제품 매출만 가져오기
'''
#매출 정보 가져오기
t_revenue_by_mitem_df = query2athena(q)

In [49]:
cols = [
    'sales_quantity',
    'total_revenue',
    'product_sales_revenue',
    'net_revenue'
]

#숫자형 컬럼들 데이터 타입 변경
t_revenue_by_mitem_df[cols] = t_revenue_by_mitem_df[cols].apply(pd.to_numeric, errors='coerce')

In [50]:
#제품 매출 없는 데이터 제거
t_revenue_by_mitem_df = t_revenue_by_mitem_df[t_revenue_by_mitem_df['product_sales_revenue'] != 0]

In [52]:
# 문자열 정리
closure_tmp = closure_df.copy()
closure_tmp['ancestor_code'] = closure_tmp['ancestor_code'].astype(str).str.strip()
closure_tmp['child_code']    = closure_tmp['child_code'].astype(str).str.strip()

# (ancestor, child) 중복을 "합산"으로 1행으로 압축
closure_uc = (
    closure_tmp
    .groupby(['ancestor_code', 'child_code'], as_index=False)['child_spqty']
    .sum()
)

# child_code_list (정렬 고정)
child_code_map = (
    closure_uc
    .groupby('ancestor_code')['child_code']
    .apply(list)  # 아래 sort에서 이미 정렬되게 만들 거라 list로 OK
)

# child_spqty_list (child_code_list와 같은 순서 보장)
closure_sorted = closure_uc.sort_values(['ancestor_code', 'child_code'])

child_code_map = (
    closure_sorted
    .groupby('ancestor_code')['child_code']
    .apply(list)
)

child_spqty_map = (
    closure_sorted
    .groupby('ancestor_code')['child_spqty']
    .apply(list)
)

# revenue df 필터 + 매핑
rev = t_revenue_by_mitem_df.copy()
rev['mitem_code'] = rev['mitem_code'].astype(str).str.strip()

rev_filtered = rev[rev['mitem_code'].isin(child_code_map.index)].copy()
rev_filtered['child_code_list']  = rev_filtered['mitem_code'].map(child_code_map)
rev_filtered['child_spqty_list'] = rev_filtered['mitem_code'].map(child_spqty_map)

In [53]:
# 동일 제품들 판매량 및 매출 합산
sum_cols = ['sales_quantity','total_revenue','product_sales_revenue','net_revenue']

tmp = rev_filtered.copy()

# (child, qty) 페어 키 생성 (정렬 + 라운딩)
def make_child_qty_key(child_list, qty_list, ndigits=10):
    if not (isinstance(child_list, list) and isinstance(qty_list, list)):
        return tuple()
    if len(child_list) != len(qty_list):
        return tuple()  # 데이터 깨진 케이스 방어
    pairs = []
    for c, q in zip(child_list, qty_list):
        c = str(c).strip()
        q = pd.to_numeric(q, errors='coerce')
        if pd.isna(q):
            q = 0
        pairs.append((c, round(float(q), ndigits)))
    return tuple(sorted(pairs))

tmp['child_qty_key'] = tmp.apply(
    lambda r: make_child_qty_key(r['child_code_list'], r['child_spqty_list'], ndigits=10),
    axis=1
)

for c in sum_cols:
    tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
tmp[sum_cols] = tmp[sum_cols].fillna(0)

group_cols = [
    'mitem_code','mitem_name','customer_code','customer_name',
    'currency','category','unit','forml_code','forml_name','base_time',
    'child_qty_key'
]

rev_grouped = (
    tmp.groupby(group_cols, as_index=False)[sum_cols].sum()
)

In [54]:
#숫자 소수 둘쨰자리로 통일
cols = ['total_revenue','product_sales_revenue','net_revenue','sales_quantity']
rev_grouped[cols] = rev_grouped[cols].round(2)

## 4. 원료-매출 베이스 테이블 생성

In [55]:
# 0) t_user_raw_df에 raw_cd 중복이 있으면 먼저 제거 (merge row 폭증 방지)
t_user_raw_df_uniq = t_user_raw_df.drop_duplicates(subset=['raw_cd']).copy()

# 1) rev_grouped -> 원료 1개 = 1행 형태로 풀기
rows = []
for _, r in rev_grouped.iterrows():
    raw_list = r['child_qty_key']  # ((raw_cd, ratio), ...) 형태

    # NaN/None 방어
    if raw_list is None or (isinstance(raw_list, float) and pd.isna(raw_list)):
        continue

    for raw_cd, ratio in raw_list:
        rows.append({
            # 원료 정보
            'raw_cd': raw_cd,
            'raw_ratio': ratio,

            # 벌크/제품 정보
            'mitem_code': r['mitem_code'],
            'mitem_name': r['mitem_name'],
            'customer_code': r['customer_code'],
            'customer_name': r['customer_name'],
            'category': r['category'],
            'forml_code': r['forml_code'],
            'forml_name': r['forml_name'],
            'base_time': r['base_time'],

            # 매출
            'total_revenue': r['total_revenue'],
            'product_sales_revenue': r['product_sales_revenue'],
            'net_revenue': r['net_revenue'],
        })

base_raw_sales_df = pd.DataFrame(rows)

# 2) raw_cd 기준으로 t_user_raw_df 붙이기 + 정렬(raw_cd, mitem_code, base_time 오름차순)
base_raw_sales_df = (
    base_raw_sales_df
    .merge(t_user_raw_df_uniq, on='raw_cd', how='left')
    .sort_values(['raw_cd', 'mitem_code', 'base_time'], ascending=[True, True, True])
    .reset_index(drop=True)
)

In [56]:
base_raw_sales_df.head()

,raw_cd,raw_ratio,mitem_code,mitem_name,customer_code,customer_name,category,forml_code,forml_name,base_time,total_revenue,product_sales_revenue,net_revenue,raw_nm,mmsta,raw_user_id
0,6034078,1.000000e-04,9HCM0000411,(비건)웅진휴캄수분결토너패드170G(일반용),104746,주식회사 웅진휴캄,FERT,11S1009,마스크 패드,2025-01-01,18730000.0,18730000.0,8200000.0,비타C브라이트닝바이옴(웅진릴리에뜨전용)(EVE비건),NO,112220050
1,6034078,1.000000e-02,9HCM0006610,(비건)웅진릴리에뜨휴캄솔싹투명마스크(10매)(브이라벨),104746,주식회사 웅진휴캄,FERT,11S1004,마스크시트,2025-01-01,13414599.0,13414599.0,6709485.0,비타C브라이트닝바이옴(웅진릴리에뜨전용)(EVE비건),NO,112220050
2,6034180,1.000000e+01,9DKP0028510,(단종)동국제약센텔리안24엑스퍼트마데카쏙앰플38ML◈,100234,동국제약 (주),FERT,11S0701,에센스,2025-01-01,39644640.0,39644640.0,21742560.0,센텔리안24탄력바이옴(W)(동국제약전용),NO,112220050
3,6036801,1.000000e+00,9GLR0006010,글로우레시피아보카도세라마이드모이스처베리어클렌저150ML,101135,Glow Recipe,FERT,11S0202,폼클렌져,2025-02-01,321686735.0,321686735.0,251536980.0,YOUTHYOME LIPOSOME,NO,112210016
4,6036950,1.000000e-07,9CBK0987110,AHC캡처솔루션시그니처모이스트앰플50ML(25AD)(용매분리),100138,(주)카버코리아,FERT,11S0701,에센스,2025-02-01,37715496.0,37715496.0,10319400.0,UNICLAY BIOTECH INGREDIENT,NO,112210016


In [57]:
col_order = [

    # 원료
    'raw_cd',
    'raw_nm',
    'raw_ratio',

    # 제품/벌크
    'mitem_code',
    'mitem_name',
    'category',
    'forml_code',
    'forml_name',

    # 고객
    'customer_code',
    'customer_name',

    # 시점
    'base_time',

    # 매출
    'total_revenue',
    'product_sales_revenue',
    'net_revenue'
]

# 컬럼 재정렬
base_raw_sales_df = base_raw_sales_df[col_order]

In [58]:
base_raw_sales_df.head()

,raw_cd,raw_nm,raw_ratio,mitem_code,mitem_name,category,forml_code,forml_name,customer_code,customer_name,base_time,total_revenue,product_sales_revenue,net_revenue
0,6034078,비타C브라이트닝바이옴(웅진릴리에뜨전용)(EVE비건),1.000000e-04,9HCM0000411,(비건)웅진휴캄수분결토너패드170G(일반용),FERT,11S1009,마스크 패드,104746,주식회사 웅진휴캄,2025-01-01,18730000.0,18730000.0,8200000.0
1,6034078,비타C브라이트닝바이옴(웅진릴리에뜨전용)(EVE비건),1.000000e-02,9HCM0006610,(비건)웅진릴리에뜨휴캄솔싹투명마스크(10매)(브이라벨),FERT,11S1004,마스크시트,104746,주식회사 웅진휴캄,2025-01-01,13414599.0,13414599.0,6709485.0
2,6034180,센텔리안24탄력바이옴(W)(동국제약전용),1.000000e+01,9DKP0028510,(단종)동국제약센텔리안24엑스퍼트마데카쏙앰플38ML◈,FERT,11S0701,에센스,100234,동국제약 (주),2025-01-01,39644640.0,39644640.0,21742560.0
3,6036801,YOUTHYOME LIPOSOME,1.000000e+00,9GLR0006010,글로우레시피아보카도세라마이드모이스처베리어클렌저150ML,FERT,11S0202,폼클렌져,101135,Glow Recipe,2025-02-01,321686735.0,321686735.0,251536980.0
4,6036950,UNICLAY BIOTECH INGREDIENT,1.000000e-07,9CBK0987110,AHC캡처솔루션시그니처모이스트앰플50ML(25AD)(용매분리),FERT,11S0701,에센스,100138,(주)카버코리아,2025-02-01,37715496.0,37715496.0,10319400.0


In [ ]:
# 동일 제품끼리 그룹화를 위한 제품명 정규화
df = base_raw_sales_df.copy()

# 닫힌 괄호 블록 전부 제거 (모든 괄호 타입)
ALL_CLOSED_BRACKETS_ANYWHERE = re.compile(
    r"[\(\[\{（【［<][^)\]\}）】］>]*[\)\]\}）】］>]"
)

PATTERNS = [
    ALL_CLOSED_BRACKETS_ANYWHERE,
    re.compile(r"\s*#.+$"),                          # 색상/쉐이드
    re.compile(r"\s*\d+(?:\.\d+)?\s*(?:ml|l|g|mg)\b", re.I),  # 용량
    re.compile(r"\s*\d+\s*\+\s*\d+\s*"),             # 번들 (1+1 등)
    re.compile(r"(기획세트|기획|세트|더블|트리플|\d+\s*개입|\d+\s*개|\d+\s*매|\d+\s*입)$"),
]

SPACE_RE = re.compile(r"\s+")


def normalize_name(name: str) -> str:
    if pd.isna(name):
        return name

    s = str(name).strip()
    prev = None

    while prev != s:
        prev = s
        for pat in PATTERNS:
            s = pat.sub(" ", s).strip()
        s = SPACE_RE.sub(" ", s).strip()

    s = re.sub(r"\s+", "", s)
    return s


df["product_name"] = df["mitem_name"].apply(normalize_name)

In [60]:
df.head()

,raw_cd,raw_nm,raw_ratio,mitem_code,mitem_name,category,forml_code,forml_name,customer_code,customer_name,base_time,total_revenue,product_sales_revenue,net_revenue,product_name
0,6034078,비타C브라이트닝바이옴(웅진릴리에뜨전용)(EVE비건),1.000000e-04,9HCM0000411,(비건)웅진휴캄수분결토너패드170G(일반용),FERT,11S1009,마스크 패드,104746,주식회사 웅진휴캄,2025-01-01,18730000.0,18730000.0,8200000.0,웅진휴캄수분결토너패드
1,6034078,비타C브라이트닝바이옴(웅진릴리에뜨전용)(EVE비건),1.000000e-02,9HCM0006610,(비건)웅진릴리에뜨휴캄솔싹투명마스크(10매)(브이라벨),FERT,11S1004,마스크시트,104746,주식회사 웅진휴캄,2025-01-01,13414599.0,13414599.0,6709485.0,웅진릴리에뜨휴캄솔싹투명마스크
2,6034180,센텔리안24탄력바이옴(W)(동국제약전용),1.000000e+01,9DKP0028510,(단종)동국제약센텔리안24엑스퍼트마데카쏙앰플38ML◈,FERT,11S0701,에센스,100234,동국제약 (주),2025-01-01,39644640.0,39644640.0,21742560.0,동국제약센텔리안24엑스퍼트마데카쏙앰플◈
3,6036801,YOUTHYOME LIPOSOME,1.000000e+00,9GLR0006010,글로우레시피아보카도세라마이드모이스처베리어클렌저150ML,FERT,11S0202,폼클렌져,101135,Glow Recipe,2025-02-01,321686735.0,321686735.0,251536980.0,글로우레시피아보카도세라마이드모이스처베리어클렌저
4,6036950,UNICLAY BIOTECH INGREDIENT,1.000000e-07,9CBK0987110,AHC캡처솔루션시그니처모이스트앰플50ML(25AD)(용매분리),FERT,11S0701,에센스,100138,(주)카버코리아,2025-02-01,37715496.0,37715496.0,10319400.0,AHC캡처솔루션시그니처모이스트앰플


## 5. Summary 생성 & Excel 출력

In [ ]:
OUTPUT_XLSX = OUTPUT_FILE

# =========================
# 컬럼 후보
# =========================
COL_CAND = {
    "month": ["base_month", "base_time", "month", "ym", "yyyymm"],
    "revenue": ["revenue", "sales_revenue", "net_revenue", "product_sales_revenue"],  # total_revenue 제외
    "raw_cd": ["raw_cd", "matnr"],
    "raw_nm": ["raw_nm", "raw_name"],
    "raw_ratio": ["raw_ratio", "ratio"],
    "mitem_code": ["mitem_code", "mitem"],
    "mitem_name": ["mitem_name"],
    "customer_code": ["customer_code"],
    "customer_name": ["customer_name"],
    "product_name": ["product_name"],  # 제품구분
    "forml_code": ["forml_code", "formulation_code", "form_code"],
    "forml_name": ["forml_name", "formulation_name", "form_name"],
}


def pick_col(df, key, required=False):
    for c in COL_CAND[key]:
        if c in df.columns:
            return c
    if required:
        raise ValueError(
            f"{key} 컬럼 없음 (후보={COL_CAND[key]}) / 현재컬럼={list(df.columns)}"
        )
    return None


def normalize_month(s: pd.Series) -> pd.Series:
    dt = pd.to_datetime(s, errors="coerce")
    if dt.notna().any():
        return dt.dt.to_period("M").astype(str)

    ss = s.astype(str).str.strip()
    mask = ss.str.fullmatch(r"\d{6}")
    out = ss.copy()
    out.loc[mask] = out.loc[mask].str[:4] + "-" + out.loc[mask].str[4:]
    return out


def add_net_margin_ratio(df_, prod_col="product_sales_revenue_sum", net_col="net_revenue_sum", out_col="net_margin"):
    """순이익률 = net / product (0~1 비율)"""
    df_[out_col] = np.where(
        pd.to_numeric(df_[prod_col], errors="coerce").fillna(0) > 0,
        pd.to_numeric(df_[net_col], errors="coerce").fillna(0) / pd.to_numeric(df_[prod_col], errors="coerce").fillna(0),
        np.nan,
    )
    return df_


# =========================
# 1) 로드
# =========================
# NOTE: 이 스크립트는 "df"가 이미 DataFrame으로 존재한다고 가정합니다.
# 혹시 df에 _컬럼이 남아있을 수 있으니, 저장용 원본에서 제거(안전장치)
df = df.drop(columns=[c for c in df.columns if str(c).startswith("_")], errors="ignore")

raw = df.copy()

c_month = pick_col(raw, "month", required=True)

# 기준 매출(막대): product_sales_revenue 우선
c_rev = "product_sales_revenue" if "product_sales_revenue" in raw.columns else pick_col(raw, "revenue", required=True)

c_raw_cd = pick_col(raw, "raw_cd", required=True)
c_raw_nm = pick_col(raw, "raw_nm", required=True)
c_raw_ratio = pick_col(raw, "raw_ratio", required=True)

c_mitem_code = pick_col(raw, "mitem_code")  # product_unique_cnt용
c_cust_code = pick_col(raw, "customer_code", required=True)
c_cust_name = pick_col(raw, "customer_name", required=True)

c_product_name = pick_col(raw, "product_name", required=True)
c_forml_code = pick_col(raw, "forml_code", required=True)
c_forml_name = pick_col(raw, "forml_name", required=True)

prod_col = "product_sales_revenue" if "product_sales_revenue" in raw.columns else None
net_col = "net_revenue" if "net_revenue" in raw.columns else None
if prod_col is None:
    raise ValueError("product_sales_revenue 컬럼이 없습니다.")
if net_col is None:
    raise ValueError("net_revenue 컬럼이 없습니다.")

# Raw에 절대 저장하지 않는 작업용 DF
df_work = raw.copy()
df_work["_month"] = normalize_month(df_work[c_month])
df_work["_prod_rev"] = pd.to_numeric(df_work[prod_col], errors="coerce").fillna(0)
df_work["_net_rev"] = pd.to_numeric(df_work[net_col], errors="coerce").fillna(0)
df_work["_rev"] = pd.to_numeric(df_work[c_rev], errors="coerce").fillna(0)


# =========================
# 2) Summary 생성
# =========================

# Summary_총매출 (+ 순이익률)
sum_total = pd.DataFrame(
    {
        "product_sales_revenue_sum": [df_work["_prod_rev"].sum()],
        "net_revenue_sum": [df_work["_net_rev"].sum()],
        "mitem_code_uniq": [raw[c_mitem_code].nunique() if c_mitem_code else np.nan],
        "raw_cd_uniq": [raw[c_raw_cd].nunique()],
        "customer_code_uniq": [raw[c_cust_code].nunique()],
    }
)
sum_total = add_net_margin_ratio(sum_total, out_col="net_margin")
# 컬럼 순서: net 옆에 순이익률
sum_total = sum_total[
    ["product_sales_revenue_sum", "net_revenue_sum", "net_margin", "mitem_code_uniq", "raw_cd_uniq", "customer_code_uniq"]
]

# Summary_월별 (+ 순이익률)
sum_month = (
    df_work.groupby("_month", dropna=False)
    .agg(
        product_sales_revenue_sum=("_prod_rev", "sum"),
        net_revenue_sum=("_net_rev", "sum"),
    )
    .reset_index()
    .sort_values("_month")
)
sum_month = add_net_margin_ratio(sum_month, out_col="net_margin")
# 컬럼 순서 정리
sum_month = sum_month[["_month", "product_sales_revenue_sum", "net_revenue_sum", "net_margin"]]

# 공통: contribution 컬럼명을 더 직관적으로
CONTRIB_COL = "revenue_share"  # (= product_sales_revenue_sum 기준 비중)

# Summary_원료 (요청 컬럼/순서)
sum_raw = (
    df_work.groupby([c_raw_cd, c_raw_nm], dropna=False)
    .agg(
        product_sales_revenue_sum=("_prod_rev", "sum"),
        net_revenue_sum=("_net_rev", "sum"),
        product_unique_cnt=(c_mitem_code, "nunique") if c_mitem_code else ("_month", "size"),
    )
    .reset_index()
)
base_sum = float(sum_raw["product_sales_revenue_sum"].sum()) if len(sum_raw) else 0.0
sum_raw[CONTRIB_COL] = (sum_raw["product_sales_revenue_sum"] / base_sum) if base_sum > 0 else 0
sum_raw = add_net_margin_ratio(sum_raw, out_col="net_margin")
sum_raw = sum_raw.sort_values("product_sales_revenue_sum", ascending=False)
sum_raw = sum_raw[[c_raw_cd, c_raw_nm, "product_sales_revenue_sum", "net_revenue_sum", "net_margin", CONTRIB_COL, "product_unique_cnt"]]

# Summary_고객 (요청 컬럼/순서)
sum_cust = (
    df_work.groupby([c_cust_code, c_cust_name], dropna=False)
    .agg(
        product_sales_revenue_sum=("_prod_rev", "sum"),
        net_revenue_sum=("_net_rev", "sum"),
        product_unique_cnt=(c_mitem_code, "nunique") if c_mitem_code else ("_month", "size"),
    )
    .reset_index()
)
base_sum = float(sum_cust["product_sales_revenue_sum"].sum()) if len(sum_cust) else 0.0
sum_cust[CONTRIB_COL] = (sum_cust["product_sales_revenue_sum"] / base_sum) if base_sum > 0 else 0
sum_cust = add_net_margin_ratio(sum_cust, out_col="net_margin")
sum_cust = sum_cust.sort_values("product_sales_revenue_sum", ascending=False)
sum_cust = sum_cust[[c_cust_code, c_cust_name, "product_sales_revenue_sum", "net_revenue_sum", "net_margin", CONTRIB_COL, "product_unique_cnt"]]

# Summary_고객_원료 (요청 컬럼/순서)
sum_cust_raw = (
    df_work.groupby([c_cust_code, c_cust_name, c_raw_cd, c_raw_nm], dropna=False)
    .agg(
        product_sales_revenue_sum=("_prod_rev", "sum"),
        net_revenue_sum=("_net_rev", "sum"),
        product_unique_cnt=(c_mitem_code, "nunique") if c_mitem_code else ("_month", "size"),
    )
    .reset_index()
)
base_sum = float(sum_cust_raw["product_sales_revenue_sum"].sum()) if len(sum_cust_raw) else 0.0
sum_cust_raw[CONTRIB_COL] = (sum_cust_raw["product_sales_revenue_sum"] / base_sum) if base_sum > 0 else 0
sum_cust_raw = add_net_margin_ratio(sum_cust_raw, out_col="net_margin")
sum_cust_raw = sum_cust_raw.sort_values("product_sales_revenue_sum", ascending=False)
sum_cust_raw = sum_cust_raw[
    [c_cust_code, c_cust_name, c_raw_cd, c_raw_nm, "product_sales_revenue_sum", "net_revenue_sum", "net_margin", CONTRIB_COL, "product_unique_cnt"]
]

# Summary_고객_제품 (요청 컬럼/순서)
sum_cust_prod = (
    df_work.groupby([c_cust_code, c_cust_name, c_product_name], dropna=False)
    .agg(
        product_sales_revenue_sum=("_prod_rev", "sum"),
        net_revenue_sum=("_net_rev", "sum"),
        product_unique_cnt=(c_mitem_code, "nunique") if c_mitem_code else ("_month", "size"),
    )
    .reset_index()
)
base_sum = float(sum_cust_prod["product_sales_revenue_sum"].sum()) if len(sum_cust_prod) else 0.0
sum_cust_prod[CONTRIB_COL] = (sum_cust_prod["product_sales_revenue_sum"] / base_sum) if base_sum > 0 else 0
sum_cust_prod = add_net_margin_ratio(sum_cust_prod, out_col="net_margin")
sum_cust_prod = sum_cust_prod.sort_values("product_sales_revenue_sum", ascending=False)
sum_cust_prod = sum_cust_prod[
    [c_cust_code, c_cust_name, c_product_name, "product_sales_revenue_sum", "net_revenue_sum", "net_margin", CONTRIB_COL, "product_unique_cnt"]
]

# Summary_고객_제형 (요청 컬럼/순서)
sum_cust_form = (
    df_work.groupby([c_cust_code, c_cust_name, c_forml_code, c_forml_name], dropna=False)
    .agg(
        product_sales_revenue_sum=("_prod_rev", "sum"),
        net_revenue_sum=("_net_rev", "sum"),
        product_unique_cnt=(c_mitem_code, "nunique") if c_mitem_code else ("_month", "size"),
    )
    .reset_index()
)
base_sum = float(sum_cust_form["product_sales_revenue_sum"].sum()) if len(sum_cust_form) else 0.0
sum_cust_form[CONTRIB_COL] = (sum_cust_form["product_sales_revenue_sum"] / base_sum) if base_sum > 0 else 0
sum_cust_form = add_net_margin_ratio(sum_cust_form, out_col="net_margin")
sum_cust_form = sum_cust_form.sort_values("product_sales_revenue_sum", ascending=False)
sum_cust_form = sum_cust_form[
    [c_cust_code, c_cust_name, c_forml_code, c_forml_name, "product_sales_revenue_sum", "net_revenue_sum", "net_margin", CONTRIB_COL, "product_unique_cnt"]
]

# Raw_product (정렬: raw_cd, customer_code, product_name 오름차순)
raw_product = (
    df_work.groupby(
        [c_raw_cd, c_raw_nm, c_raw_ratio, c_product_name, c_forml_code, c_forml_name, c_cust_code, c_cust_name],
        dropna=False,
    )
    .agg(
        product_sales_revenue_sum=("_prod_rev", "sum"),
        net_revenue_sum=("_net_rev", "sum"),
    )
    .reset_index()
    .sort_values([c_raw_cd, c_cust_code, c_product_name], ascending=[True, True, True])
)
raw_product = add_net_margin_ratio(raw_product, out_col="net_margin")
# 보기 좋게(매출/이익/순이익률을 뒤에 모으기)
raw_product = raw_product[
    [c_raw_cd, c_raw_nm, c_raw_ratio, c_cust_code, c_cust_name, c_product_name, c_forml_code, c_forml_name,
     "product_sales_revenue_sum", "net_revenue_sum", "net_margin"]
]


# =========================
# 3) Excel 저장 + DataBar
# =========================
with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as writer:
    wb = writer.book

    fmt_int = wb.add_format({"num_format": "#,##0"})
    fmt_pct = wb.add_format({"num_format": "0.0%"})
    fmt_header = wb.add_format({"bold": True, "bg_color": "#F2F2F2"})

    def add_data_bar(ws, df_, col_name, bar_color="#5B9BD5"):
        if col_name not in df_.columns:
            return
        s = pd.to_numeric(df_[col_name], errors="coerce").fillna(0)
        mx = float(s.max()) if len(s) else 0.0
        if mx <= 0:
            return
        col = df_.columns.get_loc(col_name)
        ws.conditional_format(
            1, col, len(df_), col,
            {
                "type": "data_bar",
                "bar_color": bar_color,
                "min_type": "num", "min_value": 0,
                "max_type": "num", "max_value": mx
            }
        )

    def base_format(sheet_name, df_):
        ws = writer.sheets[sheet_name]

        for i, c in enumerate(df_.columns):
            try:
                base_w = max(df_[c].astype(str).map(len).max(), len(str(c))) + 2
            except Exception:
                base_w = len(str(c)) + 2

            # 길게 보장할 컬럼(기존 룰 유지)
            if c in ["mitem_name", "product_name"]:
                w = max(base_w, 50)
                w = min(w, 120)
            else:
                w = min(base_w, 70)

            ws.set_column(i, i, w)

        ws.freeze_panes(1, 0)

        if len(df_) > 0 and len(df_.columns) > 0:
            ws.autofilter(0, 0, len(df_), len(df_.columns) - 1)
            ws.conditional_format(
                0, 0, 0, len(df_.columns) - 1,
                {"type": "no_errors", "format": fmt_header}
            )

        return ws

    # =========================
    # 시트 작성 (Raw는 원본만)
    # =========================
    # Raw 시트에서 total_revenue 제거
    if "total_revenue" in raw.columns:
        raw = raw.drop(columns=["total_revenue"])
    
    raw.to_excel(writer, sheet_name="Raw", index=False)
    raw_product.to_excel(writer, sheet_name="Raw_product", index=False)

    sum_total.to_excel(writer, sheet_name="Summary_총매출", index=False)
    sum_month.to_excel(writer, sheet_name="Summary_월별", index=False)
    sum_raw.to_excel(writer, sheet_name="Summary_원료", index=False)
    sum_cust.to_excel(writer, sheet_name="Summary_고객", index=False)
    sum_cust_raw.to_excel(writer, sheet_name="Summary_고객_원료", index=False)
    sum_cust_prod.to_excel(writer, sheet_name="Summary_고객_제품", index=False)
    sum_cust_form.to_excel(writer, sheet_name="Summary_고객_제형", index=False)

    # Raw: 천단위 + 막대
    ws = base_format("Raw", raw)
    raw_money_cols = [c for c in ["product_sales_revenue", "net_revenue"] if c in raw.columns]
    for col in raw_money_cols:
        idx = raw.columns.get_loc(col)
        ws.set_column(idx, idx, 16, fmt_int)
    if c_rev in raw.columns:
        add_data_bar(ws, raw, c_rev)

    # Raw_product: 천단위 + DataBar
    ws = base_format("Raw_product", raw_product)
    for col in ["product_sales_revenue_sum", "net_revenue_sum"]:
        if col in raw_product.columns:
            idx = raw_product.columns.get_loc(col)
            ws.set_column(idx, idx, 18, fmt_int)
            add_data_bar(ws, raw_product, col)
    if "net_margin" in raw_product.columns:
        idx = raw_product.columns.get_loc("net_margin")
        ws.set_column(idx, idx, 12, fmt_pct)

    # Summary_총매출
    ws = base_format("Summary_총매출", sum_total)
    for col in ["product_sales_revenue_sum", "net_revenue_sum"]:
        idx = sum_total.columns.get_loc(col)
        ws.set_column(idx, idx, 22, fmt_int)
    if "net_margin" in sum_total.columns:
        idx = sum_total.columns.get_loc("net_margin")
        ws.set_column(idx, idx, 12, fmt_pct)

    # Summary_월별
    ws = base_format("Summary_월별", sum_month)
    for col in ["product_sales_revenue_sum", "net_revenue_sum"]:
        idx = sum_month.columns.get_loc(col)
        ws.set_column(idx, idx, 22, fmt_int)
        add_data_bar(ws, sum_month, col)
    if "net_margin" in sum_month.columns:
        idx = sum_month.columns.get_loc("net_margin")
        ws.set_column(idx, idx, 12, fmt_pct)

    # Summary_원료
    ws = base_format("Summary_원료", sum_raw)
    for col in ["product_sales_revenue_sum", "net_revenue_sum"]:
        idx = sum_raw.columns.get_loc(col)
        ws.set_column(idx, idx, 18, fmt_int)
        add_data_bar(ws, sum_raw, col)
    for col in ["net_margin", CONTRIB_COL]:
        if col in sum_raw.columns:
            idx = sum_raw.columns.get_loc(col)
            ws.set_column(idx, idx, 12, fmt_pct)

    # Summary_고객
    ws = base_format("Summary_고객", sum_cust)
    for col in ["product_sales_revenue_sum", "net_revenue_sum"]:
        idx = sum_cust.columns.get_loc(col)
        ws.set_column(idx, idx, 18, fmt_int)
        add_data_bar(ws, sum_cust, col)
    for col in ["net_margin", CONTRIB_COL]:
        idx = sum_cust.columns.get_loc(col)
        ws.set_column(idx, idx, 12, fmt_pct)

    # Summary_고객_원료
    ws = base_format("Summary_고객_원료", sum_cust_raw)
    for col in ["product_sales_revenue_sum", "net_revenue_sum"]:
        idx = sum_cust_raw.columns.get_loc(col)
        ws.set_column(idx, idx, 18, fmt_int)
        add_data_bar(ws, sum_cust_raw, col)
    for col in ["net_margin", CONTRIB_COL]:
        idx = sum_cust_raw.columns.get_loc(col)
        ws.set_column(idx, idx, 12, fmt_pct)

    # Summary_고객_제품
    ws = base_format("Summary_고객_제품", sum_cust_prod)
    for col in ["product_sales_revenue_sum", "net_revenue_sum"]:
        idx = sum_cust_prod.columns.get_loc(col)
        ws.set_column(idx, idx, 18, fmt_int)
        add_data_bar(ws, sum_cust_prod, col)
    for col in ["net_margin", CONTRIB_COL]:
        idx = sum_cust_prod.columns.get_loc(col)
        ws.set_column(idx, idx, 12, fmt_pct)

    # Summary_고객_제형
    ws = base_format("Summary_고객_제형", sum_cust_form)
    for col in ["product_sales_revenue_sum", "net_revenue_sum"]:
        idx = sum_cust_form.columns.get_loc(col)
        ws.set_column(idx, idx, 18, fmt_int)
        add_data_bar(ws, sum_cust_form, col)
    for col in ["net_margin", CONTRIB_COL]:
        idx = sum_cust_form.columns.get_loc(col)
        ws.set_column(idx, idx, 12, fmt_pct)

print("완료:", OUTPUT_XLSX)